# Somo 07 - Mfumo wa Ubunifu wa Mipango

Daftari hili linaonyesha **Mfumo wa Ubunifu wa Mipango** kwa wawakilishi wa AI kwa kutumia Mfumo wa Wakala wa Microsoft.
Utajifunza jinsi ya kugawanya ombi tata la usafiri kuwa kazi ndogo zilizopangwa, kuziwapa mawakala maalum,
na kutekeleza mpango unaotokana — yote kwa kutumia matokeo yaliyopangwa yanayotumia mifano ya Pydantic.


## Usanidi


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os, asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Ugawaji wa Kazi

Ugawaji wa kazi ni msingi wa mfano wa kubuni wa upangaji. Badala ya kumuomba wakala mmoja kushughulikia ombi tata kuanzia mwanzo hadi mwisho,
tunagawanya tatizo kuwa **vikundi vidogo vya kazi** vilivyoeleweka vyema.
Kila kazi ndogo hupewa wakala mtaalamu (mfano, safari za ndege, hoteli, shughuli) kwa vipaumbele wazi
na mpangilio wa utegemezi.

Njia hii inatoa faida kadhaa:
- **Uwazi**: kila kazi ndogo ina jukumu moja tu.
- **Mwendo sambamba**: kazi ndogo zisizo tegemezana zinaweza kufanywa kwa wakati mmoja.
- **Uaminifu**: kushindwa kunazingirwa kwa kazi ndogo binafsi.
- **Ufuatiliaji wa bajeti**: gharama zinakadiriwa kwa kila kazi ndogo na kujumlishwa.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## Kuunda Wakala wa Mipango na Matokeo Yaliyopangwa

Wakala wa mipango hufanya kazi kama **mkurugenzi wa ofisi ya mapokezi**. Kwa ombi la usafiri la ngazi ya juu
huunda `TravelPlan` iliyopangwa — akigawa ombi hilo katika kazi ndogo ndogo, kuweka kipaumbele,
na kubaini utegemezi ili mhudumu au tabaka la utekelezaji wafanye kazi hiyo.


In [ ]:
planning_agent = client.as_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    options={"response_format": TravelPlan}
)
if result:
    plan = result.value
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## Kutekeleza Mpango kwa Vifaa Maalum

Mara wakala wa dawati la mbele atakapotoa mpango uliopangwa, **wakala wa concierge** hutekeleza.
Kila chombo maalum hushughulikia aina moja ya kazi ndogo ndogo (ndege, hoteli, shughuli). Concierge
huenda kupitia kazi ndogo ndogo za mpango kwa mpangilio wa utegemezi na kutuma kila moja kwa
chombo kinachofaa.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = client.as_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## Muhtasari

Katika somo hili ulijifunza **Mfumo wa Mipango ya Ubunifu** kwa mawakala wa AI:

1. **Ugawaji wa Kazi** — Wakala wa mipango wa dawati la mbele hugawanya ombi tata la kusafiri katika
   kazi ndogo zilizo na muundo kwa kutumia modeli za Pydantic, akiagiza kila moja kwa wakala mtaalamu kwa vipaumbele
   na utegemezi.
2. **Matokeo Yenye Muundo** — Kwa kupitisha `response_format` wakala hurudisha kitu cha `TravelPlan` kilichothibitishwa
   badala ya maandishi ya bure, na kufanya usindaji wa baadaye kuwa wa kuaminika.
3. **Utekelezaji wa Mpango** — Wakala msaidizi huenda hatua kwa hatua kupitia kazi ndogo kwa kutumia zana za kitaalamu
   (`book_flight`, `reserve_hotel`, `book_activity`) kutekeleza mpango na kuripoti matokeo.

Mfumo huu unagawa *nini cha kufanya* (mipango) kutoka *jinsi ya kukifanya* (utekelezaji), na kufanya mawakala
kuwa rahisi kubadilika, kupimika, na rahisi kuongeza vipengele.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Kionyozo**:
Hati hii imetafsiriwa kwa kutumia huduma ya tafsiri ya AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ingawa tunajitahidi kupata usahihi, tafadhali fahamu kwamba tafsiri za kiotomatiki zinaweza kuwa na makosa au upungufu wa usahihi. Hati ya asili katika lugha yake halisi inapaswa kuchukuliwa kama chanzo cha mamlaka. Kwa taarifa muhimu, tafsiri ya kitaalamu inayofanywa na binadamu inapendekezwa. Hatutojibu kwa kuelewa vibaya au tafsiri potofu zinazotokea kutokana na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
